# Practical 01 — Multisensor Satellite Vegetation Indices
**Sentinel-2 (10 m) · Landsat 8/9 (30 m) · MODIS (250 m, 500 m, 1 km)**  
**Updated v3:** ROI and results maps use Esri World Imagery to avoid the OpenStreetMap 403 error.  
**Indices:** NDVI, EVI, SAVI | **Season:** June 1–August 31, each selected year  
**Platform:** Google Colab + each student's own Google Earth Engine project and Google Drive

### Learning outcomes
By the end, you will be able to select your own **region of interest (ROI) during execution**, retrieve and quality-filter satellite imagery, calculate and interpret three vegetation indices, compare observations **within and between growing seasons**, and export numerical, graphical and geospatial results to **your Google Drive**.

### Scientific rules for comparison
- Sentinel-2 provides 10 m red/NIR/blue observations; Landsat 8/9 provides 30 m. Both use surface reflectance after correct scaling and masking.
- MODIS MOD13Q1 / MOD13A1 / MOD13A2 are **independent 16-day vegetation-index products at 250 m / 500 m / 1 km**. Their NDVI/EVI are taken from the official NASA products; SAVI is calculated from the red and NIR reflectances contained in each product. Do **not** treat the three MODIS resolutions as identical measurements on differently sized grids.
- The annual value here means the **June–August seasonal mean for a given year**: a pixelwise mean of quality-controlled observations available in that window, followed by an ROI spatial mean. It is **not** a 12-month mean. Observations are not evenly spaced and different sensors have different sampling/composite strategies.
- MODIS 16-day dates indicate **nominal composite dates** rather than single-day acquisitions; some compositing windows cross the June/August boundaries. Sensor-to-sensor differences cannot be interpreted as pure resolution effects.
- A moderately large ROI (e.g., **100–300 km²**) is recommended so that MODIS 1 km has enough pixels for a meaningful spatial average; large ROIs require more processing time.

**Directions:** First use **File → Save a copy in Drive** in Google Colab, so your completed notebook is also owned by you. Then run cells in order. Where you see `CHANGE_ME`, enter your own information. The notebook will stop rather than silently saving to a generic folder.
**Version 3 fix:** explicit Earth Engine `ee.Image` casting after `copyProperties()` and pre-processing smoke tests avoid the `AttributeError: 'Element' object has no attribute 'select'` failure.


## 1. Environment check — use Colab’s preinstalled scientific libraries

Google Colab provides Earth Engine, geemap and the scientific Python libraries needed here. **Do not bulk-upgrade the environment:** upgrading `earthengine-api`, `google-auth`, or `pandas` can introduce version conflicts. If imports fail after earlier installations, use **Runtime → Disconnect and delete runtime** to start fresh.

In [ ]:
#@title Check required packages (no installation or upgrades)
from importlib import util, metadata
NEEDED_MODULES = {'ee':'earthengine-api','geemap':'geemap','ipywidgets':'ipywidgets','numpy':'numpy','pandas':'pandas','matplotlib':'matplotlib','requests':'requests','PIL':'pillow'}
missing = [package for module,package in NEEDED_MODULES.items() if util.find_spec(module) is None]
if missing:
    raise RuntimeError('Colab runtime missing: '+', '.join(missing)+'. Start a fresh standard Colab runtime; do not bulk-upgrade google-auth or pandas.')
print('All required packages available. No installations performed.')
for package in ['earthengine-api','geemap','google-auth','pandas']:
    try:
        print(f'{package}: {metadata.version(package)}')
    except metadata.PackageNotFoundError:
        print(f'{package}: unknown')

In [ ]:
import os, re, json, math, time, warnings
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import ee
from IPython.display import display

# Using geemap.core explicitly enables the ipyleaflet drawing backend.
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
import geemap.core as geemap

plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 10, 'axes.labelsize': 11, 'axes.titlesize': 12,
})
print('Libraries imported.')

## 2. Student configuration — EDIT THIS CELL FIRST

In [ ]:
#@title Practical 1 — Student configuration
# REQUIRED: your own Google Cloud project that is registered for Earth Engine.
EE_PROJECT_ID = 'CHANGE_ME'  #@param {type:"string"} # Your Earth Engine-enabled Google Cloud project
STUDENT_TAG = 'CHANGE_ME'  #@param {type:"string"} # Your student identifier

# Choose at least two years to compare. All imagery is selected June 1–August 31.
START_YEAR = 2023
END_YEAR = 2025
FOCUS_YEAR = END_YEAR               # Year used for maps and GeoTIFF exports

# ROI selection: 'draw' (recommended), 'bbox', or 'geojson'.
ROI_METHOD = 'draw'
# Used only if ROI_METHOD = 'bbox' (west, south, east, north; decimal degrees).
BBOX = (-98.25, 49.58, -98.08, 49.73)  # Example agricultural area near Carman, MB

# Used only if ROI_METHOD = 'geojson' and a GeoJSON file is already on Colab/Drive.
ROI_GEOJSON_PATH = '/content/roi.geojson'

S2_MAX_CLOUD_PERCENT = 85           # Scene-level pre-filter; SCL provides pixel masking
MODIS_MAX_SUMMARY_QA = 1            # 0 = good only; 1 = good and marginal
S2_TIMESERIES_SCALE_M = 30          # 30 m speeds ROI statistics; maps remain native 10 m
MAKE_MAP_PNGS = True                # PNG map for each index × sensor in FOCUS_YEAR
START_GEE_GEOTIFF_EXPORTS = True    # 1 three-band GeoTIFF / sensor / year (15 for 3 years)

# All tabular/figure results are saved via the student's own Drive mount.
DRIVE_BASE_FOLDER = 'Satellite_Practicals'

if EE_PROJECT_ID.strip() in ('', 'CHANGE_ME'):
    raise ValueError('Set EE_PROJECT_ID to YOUR Earth Engine-enabled Google Cloud project.')
if STUDENT_TAG.strip() in ('', 'CHANGE_ME'):
    raise ValueError('Set STUDENT_TAG to YOUR name or student identifier.')
if not (2000 <= START_YEAR <= END_YEAR <= datetime.now().year):
    raise ValueError('Check START_YEAR and END_YEAR.')
if FOCUS_YEAR < START_YEAR or FOCUS_YEAR > END_YEAR:
    raise ValueError('FOCUS_YEAR must be within the selected years.')
STUDENT_TAG = re.sub(r'[^A-Za-z0-9_-]+', '_', STUDENT_TAG).strip('_')
if not STUDENT_TAG:
    raise ValueError('Invalid STUDENT_TAG.')
YEARS = list(range(START_YEAR, END_YEAR + 1))
INDICES = ('NDVI', 'EVI', 'SAVI')
print(f'Student: {STUDENT_TAG} | Years: {YEARS} | June 1–August 31')

## 3. Authenticate your own Earth Engine project and mount your own Google Drive

In [ ]:
# Colab's Drive authorization saves CSV/PNG/JSON/ZIP files to the currently signed-in student's Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    MYDRIVE = Path('/content/drive/MyDrive')
except ImportError:
    # Outside Colab: choose a local directory rather than writing to someone else's Drive.
    MYDRIVE = Path.home() / 'MyDrive'
    MYDRIVE.mkdir(parents=True, exist_ok=True)
    print('Outside Colab: outputs go to local folder', MYDRIVE)

# Authentication uses the student's Google account and Earth Engine project.
ee.Authenticate()
ee.Initialize(project=EE_PROJECT_ID)
print('Earth Engine initialized with project:', EE_PROJECT_ID)

OUTPUT_DIR = MYDRIVE / DRIVE_BASE_FOLDER / 'Practical_01' / STUDENT_TAG
FIG_DIR = OUTPUT_DIR / 'figures'
CSV_DIR = OUTPUT_DIR / 'tables'
MAP_DIR = OUTPUT_DIR / 'map_png'
for folder in (OUTPUT_DIR, FIG_DIR, CSV_DIR, MAP_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Earth Engine batch exports require a Drive folder NAME, not a nested path.
# This is a separate top-level folder in the student's own Google Drive.
GEE_EXPORT_DRIVE_FOLDER = f'P01_{STUDENT_TAG}_GeoTIFF'
print('CSV/PNG output folder:', OUTPUT_DIR)
print('Earth Engine GeoTIFF folder (Drive top level):', GEE_EXPORT_DRIVE_FOLDER)

## 4. Select your ROI **while the notebook is running**

**Option A — Draw (recommended)**: Run the map cell, zoom to your chosen area, use the polygon/rectangle drawing tool, and draw one or more polygons. The map uses **Esri World Imagery**, not the OpenStreetMap server that can return HTTP 403 in Google Colab. **Then** run the next cell to lock the ROI. This uses the two-way `ipyleaflet` widget. If your particular Colab runtime does not display the drawing control, use Option B or C; the notebook does not assume you drew a shape if none was captured.

**Option B — Bounding box**: Change `ROI_METHOD = 'bbox'` in Section 2 and edit `BBOX = (west, south, east, north)` using coordinates of your chosen agricultural region.

**Option C — GeoJSON**: Upload a `.geojson` file via Colab's file browser or put it on your Drive, set `ROI_METHOD = 'geojson'`, and enter its absolute path in `ROI_GEOJSON_PATH`. Polygons and MultiPolygons are supported.

**Troubleshooting basemaps:** If the map shows `403 Access blocked`, run the latest map cell (the older map may still use OpenStreetMap). If Esri imagery is unavailable on your network too, switch `basemap='Esri.WorldImagery'` to `basemap='CartoDB.Positron'` in that cell and run it again. The ROI drawing tools are independent from the background tiles.

**Recommended**: a compact agricultural ROI roughly 10–20 km across. Avoid a single field when comparing with 1 km MODIS pixels.

In [ ]:
# Interactive ROI selection (Colab / ipyleaflet).
# IMPORTANT: Do NOT use the default OpenStreetMap tiles. Some Colab sessions
# receive a 403 Access blocked from the OpenStreetMap tile server.
# Esri World Imagery is a separate satellite basemap provider and does not
# send tile requests to the blocked OSM server.
roi_map = geemap.Map(center=(49.65, -98.18), zoom=10,
                     basemap='Esri.WorldImagery')
print('Draw a polygon or rectangle on the satellite map, then run the NEXT cell.')
print('If the imagery tiles are also blocked on your network, change the basemap')
print("above to 'CartoDB.Positron' and rerun this cell; your GEE analysis is unchanged.")
display(roi_map)

In [ ]:
def load_geojson_roi(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    if data.get('type') == 'FeatureCollection':
        geoms = [ee.Geometry(ft['geometry']) for ft in data.get('features', [])
                 if ft.get('geometry') is not None]
        if not geoms:
            raise ValueError('GeoJSON FeatureCollection has no geometries.')
        return ee.FeatureCollection([ee.Feature(g) for g in geoms]).geometry()
    if data.get('type') == 'Feature':
        return ee.Geometry(data['geometry'])
    if data.get('type') in ('Polygon', 'MultiPolygon', 'GeometryCollection'):
        return ee.Geometry(data)
    raise ValueError('GeoJSON must contain a Polygon/MultiPolygon or FeatureCollection.')

if ROI_METHOD == 'draw':
    draw_control = getattr(roi_map, '_draw_control', None)
    drawn_collection = getattr(draw_control, 'collection', None) if draw_control is not None else None
    if drawn_collection is None:
        raise RuntimeError('No ROI drawing detected. Draw on the map, or choose bbox/geojson in Section 2.')
    roi = ee.FeatureCollection(drawn_collection).geometry()
elif ROI_METHOD == 'bbox':
    west, south, east, north = BBOX
    if west >= east or south >= north:
        raise ValueError('BBOX order must be (west, south, east, north).')
    roi = ee.Geometry.Rectangle(list(BBOX), geodesic=False)
elif ROI_METHOD == 'geojson':
    roi = load_geojson_roi(ROI_GEOJSON_PATH)
else:
    raise ValueError('ROI_METHOD must be draw, bbox, or geojson.')

area_km2 = float(roi.area(maxError=1).divide(1e6).getInfo())
if area_km2 <= 0:
    raise ValueError('Empty ROI area.')
print(f'Selected ROI: {area_km2:,.1f} km²')
if area_km2 < 100:
    print('WARNING: fewer 1 km MODIS pixels; consider a larger ROI for reliable comparison.')
if area_km2 > 500:
    print('WARNING: large ROI; high-resolution extraction and maps may process slowly.')

roi_geojson = roi.getInfo()
with open(OUTPUT_DIR / 'student_roi.geojson', 'w', encoding='utf-8') as f:
    json.dump(roi_geojson, f, indent=2)
roi_map.add_layer(roi, {'color': 'red'}, 'Confirmed student ROI')
print('ROI saved in your Drive:', OUTPUT_DIR / 'student_roi.geojson')

## 5. Vegetation indices and quality control

Reflectance values must be scaled to approximately 0–1 **before** applying the equations.

\[
NDVI = \frac{NIR - RED}{NIR + RED}
\]
\[
EVI = 2.5\,\frac{NIR - RED}{NIR + 6\,RED - 7.5\,BLUE + 1}
\]
\[
SAVI = (1+L)\,\frac{NIR - RED}{NIR + RED + L}, \qquad L=0.5
\]

- **S2:** use B8 (NIR), B4 (red), B2 (blue), apply 0.0001 reflectance scale; keep usable SCL classes (4 vegetation, 5 bare soil, 6 water, 7 unclassified), mask invalid reflectance and problematic pixels.
- **Landsat 8/9:** use SR_B5 (NIR), SR_B4 (red), SR_B2 (blue); apply Collection 2 Level-2 `DN × 0.0000275 − 0.2`; reject fill/cloud/shadow/cirrus/snow and saturation.
- **MODIS:** use NASA vegetation-index product `NDVI`/`EVI` at `×0.0001`, and compute SAVI from corresponding `sur_refl_b01`/`sur_refl_b02` at `×0.0001`. Use `SummaryQA` 0–1 by default. MODIS EVI is taken from the product, **not recalculated with the simple equation above**.

EVI can be unstable where its denominator approaches zero. We mask near-zero denominators for the two calculated optical collections. The sensor algorithms, spectral response functions, QA and compositing remain different, so cross-sensor offsets are expected.

In [ ]:
def calc_optical_indices(image):
    """Input bands BLUE, RED, NIR in unitless surface reflectance.

    Cast ee.Element results back to ee.Image (e.g., after copyProperties()).
    """
    image = ee.Image(image)
    blue = image.select('BLUE')
    red = image.select('RED')
    nir = image.select('NIR')

    valid = (blue.gte(0).And(blue.lte(1))
             .And(red.gte(0)).And(red.lte(1))
             .And(nir.gte(0)).And(nir.lte(1)))

    ndvi_denom = nir.add(red)
    ndvi = nir.subtract(red).divide(ndvi_denom).rename('NDVI') \
        .updateMask(ndvi_denom.abs().gt(0.0001))

    evi_denom = nir.add(red.multiply(6)).subtract(blue.multiply(7.5)).add(1)
    evi = nir.subtract(red).multiply(2.5).divide(evi_denom).rename('EVI') \
        .updateMask(evi_denom.abs().gt(0.05))

    savi = nir.subtract(red).multiply(1.5).divide(nir.add(red).add(0.5)) \
        .rename('SAVI')
    indices = ee.Image.cat([ndvi, evi, savi]).updateMask(valid)
    return ee.Image(indices.copyProperties(image, ['system:time_start']))


def prep_sentinel2(img):
    scl = img.select('SCL')
    keep = (scl.eq(4).Or(scl.eq(5)).Or(scl.eq(6)).Or(scl.eq(7)))
    sr = img.select(['B2', 'B4', 'B8'], ['BLUE', 'RED', 'NIR']) \
            .multiply(0.0001).updateMask(keep)
    return calc_optical_indices(ee.Image(sr.copyProperties(img, ['system:time_start'])))


def prep_landsat(img):
    qa = img.select('QA_PIXEL')
    # QA bits 0 to 5: fill, dilated cloud, cirrus, cloud, shadow, snow.
    clean = qa.bitwiseAnd(63).eq(0).And(img.select('QA_RADSAT').eq(0))
    sr = img.select(['SR_B2', 'SR_B4', 'SR_B5'], ['BLUE', 'RED', 'NIR']) \
            .multiply(0.0000275).add(-0.2).updateMask(clean)
    return calc_optical_indices(ee.Image(sr.copyProperties(img, ['system:time_start'])))


def prep_modis(img):
    qa_ok = img.select('SummaryQA').lte(MODIS_MAX_SUMMARY_QA)
    red = img.select('sur_refl_b01').multiply(0.0001)
    nir = img.select('sur_refl_b02').multiply(0.0001)
    valid = red.gte(0).And(red.lte(1)).And(nir.gte(0)).And(nir.lte(1))
    savi = nir.subtract(red).multiply(1.5).divide(nir.add(red).add(0.5)) \
        .rename('SAVI')
    vi = img.select(['NDVI', 'EVI']).multiply(0.0001)
    combined = vi.addBands(savi).updateMask(qa_ok.And(valid))
    return ee.Image(combined.copyProperties(img, ['system:time_start']))

# Lightweight type smoke tests: catch image-versus-element errors BEFORE mapping collections.
# These only construct local Earth Engine expressions; they do not download data.
_test_s2 = ee.Image(ee.Image.constant([1000, 2000, 5000, 4])
            .rename(['B2', 'B4', 'B8', 'SCL'])
            .set('system:time_start', 1685577600000))
_test_ls = ee.Image(ee.Image.constant([10000, 11000, 14000, 0, 0])
            .rename(['SR_B2', 'SR_B4', 'SR_B5', 'QA_PIXEL', 'QA_RADSAT'])
            .set('system:time_start', 1685577600000))
_test_md = ee.Image(ee.Image.constant([0, 3000, 5000, 4000, 3500])
            .rename(['SummaryQA', 'sur_refl_b01', 'sur_refl_b02', 'NDVI', 'EVI'])
            .set('system:time_start', 1685577600000))
for _label, _result in [('Sentinel-2', prep_sentinel2(_test_s2)),
                        ('Landsat', prep_landsat(_test_ls)),
                        ('MODIS', prep_modis(_test_md))]:
    assert isinstance(_result, ee.Image), f'{_label} returned a non-image EE object.'
print('Optical indices and quality masks ready. Type tests passed for all three preprocessors.')

## 6. Build five comparable satellite/product time series

In [ ]:
SENSORS = {
    'Sentinel2_10m': {'label': 'Sentinel-2 (10 m)', 'scale': 10,
                       'id': 'COPERNICUS/S2_SR_HARMONIZED', 'cadence': 'scene'},
    'Landsat8_9_30m': {'label': 'Landsat 8/9 (30 m)', 'scale': 30,
                        'id': 'LC08 + LC09 Collection 2 L2', 'cadence': 'scene'},
    'MODIS_250m': {'label': 'MODIS (250 m)', 'scale': 250,
                   'id': 'MODIS/061/MOD13Q1', 'cadence': '16-day'},
    'MODIS_500m': {'label': 'MODIS (500 m)', 'scale': 500,
                   'id': 'MODIS/061/MOD13A1', 'cadence': '16-day'},
    'MODIS_1km': {'label': 'MODIS (1 km)', 'scale': 1000,
                  'id': 'MODIS/061/MOD13A2', 'cadence': '16-day'},
}

period_start = f'{START_YEAR}-06-01'
period_end = f'{END_YEAR}-09-01'  # exclusive end: captures all of August 31

def summer_only(ic):
    return ic.filterBounds(roi).filterDate(period_start, period_end) \
             .filter(ee.Filter.calendarRange(6, 8, 'month'))

s2 = summer_only(ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')) \
    .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', S2_MAX_CLOUD_PERCENT)) \
    .map(prep_sentinel2)

l8 = summer_only(ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')).map(prep_landsat)
l9 = summer_only(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')).map(prep_landsat)
landsat = l8.merge(l9).sort('system:time_start')

collections = {
    'Sentinel2_10m': s2,
    'Landsat8_9_30m': landsat,
    'MODIS_250m': summer_only(ee.ImageCollection('MODIS/061/MOD13Q1')).map(prep_modis),
    'MODIS_500m': summer_only(ee.ImageCollection('MODIS/061/MOD13A1')).map(prep_modis),
    'MODIS_1km': summer_only(ee.ImageCollection('MODIS/061/MOD13A2')).map(prep_modis),
}

count_rows = []
for sensor_key, ic in collections.items():
    for year in YEARS:
        n = ic.filterDate(f'{year}-06-01', f'{year}-09-01').size().getInfo()
        count_rows.append({'sensor': sensor_key, 'year': year, 'image_count': n})
        print(f'{SENSORS[sensor_key]["label"]:24s} {year}: {n:4d} images/composites')
counts_df = pd.DataFrame(count_rows)
counts_df.to_csv(CSV_DIR / 'image_counts_by_sensor_year.csv', index=False)
if counts_df['image_count'].eq(0).any():
    print('WARNING: at least one sensor-year has no scenes. Missing series will remain NaN.')

## 7. June–August mean **for each year**
Each annual output is the pixelwise arithmetic mean across all available June–August images of the same index and sensor. Next, the area-weighted interpretation is approximated using the mean of pixels inside the ROI at a scale appropriate for that product. The number of valid observations varies by pixel (cloud masking and 16-day compositing), and annual sensor means do not imply equal sampling across sensors.

In [ ]:
seasonal_images = {}  # (year, sensor_key) -> 3-band seasonal mean image
annual_rows = []

for year in YEARS:
    for sensor_key, ic in collections.items():
        year_ic = ic.filterDate(f'{year}-06-01', f'{year}-09-01')
        matching = int(counts_df.query('year == @year and sensor == @sensor_key')['image_count'].iloc[0])
        if matching == 0:
            for index_name in INDICES:
                annual_rows.append({'year': year, 'sensor': sensor_key,
                                    'sensor_label': SENSORS[sensor_key]['label'],
                                    'index': index_name, 'roi_seasonal_mean': np.nan,
                                    'image_count': 0})
            continue

        annual = year_ic.select(list(INDICES)).mean().clip(roi)
        seasonal_images[(year, sensor_key)] = annual
        native_scale = SENSORS[sensor_key]['scale']
        stats = annual.reduceRegion(
            reducer=ee.Reducer.mean(), geometry=roi, scale=native_scale,
            maxPixels=1e9, bestEffort=True, tileScale=4
        ).getInfo() or {}
        for index_name in INDICES:
            annual_rows.append({
                'year': year, 'sensor': sensor_key,
                'sensor_label': SENSORS[sensor_key]['label'],
                'index': index_name, 'roi_seasonal_mean': stats.get(index_name),
                'image_count': matching
            })
        print(f'Computed annual June–Aug mean: {year} / {SENSORS[sensor_key]["label"]}')

annual_df = pd.DataFrame(annual_rows)
annual_file = CSV_DIR / 'june_august_annual_roi_means.csv'
annual_df.to_csv(annual_file, index=False)
display(annual_df.head(15))
print('Saved:', annual_file)

## 8. Within-season time series — ROI mean for every image/composite

The temporal series shows the **spatial mean within your ROI**, calculated separately for each acquisition/composite. It is not a daily interpolated curve. S2 statistics use a default **30 m sampling scale** for processing efficiency while its exported maps use a nominal 10 m scale. Modify `S2_TIMESERIES_SCALE_M` to 10 to use nominal resolution if your ROI is small and the computation is affordable.

In [ ]:
def sample_roi_timeseries(ic, scale):
    """One reduceColumns call returns lists of image dates and ROI means.

    Image-level null ROI means are allowed; they become NaN in pandas.
    """
    def add_stats(img):
        values = img.select(list(INDICES)).reduceRegion(
            reducer=ee.Reducer.mean(), geometry=roi, scale=scale,
            maxPixels=1e9, bestEffort=True, tileScale=4
        )
        return img.set({
            'mean_NDVI': values.get('NDVI'),
            'mean_EVI': values.get('EVI'),
            'mean_SAVI': values.get('SAVI'),
        })

    cols = ['system:time_start', 'mean_NDVI', 'mean_EVI', 'mean_SAVI']
    values = ic.map(add_stats).reduceColumns(ee.Reducer.toList(4), cols) \
        .get('list').getInfo()
    return values or []

series_rows = []
for sensor_key, ic in collections.items():
    scale = S2_TIMESERIES_SCALE_M if sensor_key == 'Sentinel2_10m' else SENSORS[sensor_key]['scale']
    if not (counts_df.query('sensor == @sensor_key')['image_count'].gt(0).any()):
        continue
    print('Extracting temporal observations:', SENSORS[sensor_key]['label'])
    observations = sample_roi_timeseries(ic, scale)
    for obs in observations:
        if len(obs) != 4 or obs[0] is None:
            continue
        dt = pd.to_datetime(int(obs[0]), unit='ms', utc=True)
        if dt.month < 6 or dt.month > 8 or dt.year not in YEARS:
            continue
        for i, index_name in enumerate(INDICES, start=1):
            series_rows.append({
                'sensor': sensor_key, 'sensor_label': SENSORS[sensor_key]['label'],
                'date_utc': dt.strftime('%Y-%m-%d'), 'year': dt.year,
                'day_of_year': dt.dayofyear, 'index': index_name,
                'roi_mean': float(obs[i]) if obs[i] is not None else np.nan,
                'sampling_scale_m': scale,
            })

series_df = pd.DataFrame(series_rows, columns=[
    'sensor', 'sensor_label', 'date_utc', 'year', 'day_of_year',
    'index', 'roi_mean', 'sampling_scale_m'])
series_file = CSV_DIR / 'all_observations_june_august_roi_timeseries.csv'
series_df.sort_values(['index', 'year', 'sensor', 'date_utc']).to_csv(series_file, index=False)
print(f'{len(series_df)} sensor-index observations saved to:', series_file)
display(series_df.head(12))

## 9. Compare sensors **between years** — annual June–August ROI means

In [ ]:
colors = dict(zip(SENSORS, plt.cm.tab10(np.linspace(0, 0.8, len(SENSORS)))))
fig, axs = plt.subplots(3, 1, figsize=(10, 12), sharex=True, constrained_layout=True)
for ax, vi in zip(axs, INDICES):
    data = annual_df[annual_df['index'] == vi]
    for sensor_key, meta in SENSORS.items():
        subset = data[data['sensor'] == sensor_key].sort_values('year')
        ax.plot(subset['year'], subset['roi_seasonal_mean'], marker='o', linewidth=1.8,
                color=colors[sensor_key], label=meta['label'])
    ax.axhline(0, color='0.5', linewidth=0.7)
    ax.set_title(f'{vi} — June–August mean, by year')
    ax.set_ylabel(f'Mean {vi} (unitless)')
    ax.grid(alpha=0.25)
axs[-1].set_xticks(YEARS)
axs[-1].set_xlabel('Year')
axs[0].legend(ncol=2, frameon=False, fontsize=9, loc='best')
fig.suptitle(f'Interannual comparison of vegetation indices | ROI: {area_km2:,.1f} km²', fontsize=13)
fig_path = FIG_DIR / 'interannual_3_indices_5_products.png'
fig.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)

## 10. Compare sensors **within each growing season**

In [ ]:
# Create three figures (one/index). Each contains one panel per selected year.
# Points are at actual nominal acquisition/composite dates; no invented daily values.
for vi in INDICES:
    fig, axs = plt.subplots(len(YEARS), 1, figsize=(12, max(3.2 * len(YEARS), 5)),
                            sharex=True, constrained_layout=True)
    axs = np.atleast_1d(axs)
    for ax, year in zip(axs, YEARS):
        subset = series_df[(series_df['index'] == vi) & (series_df['year'] == year)]
        for sensor_key, meta in SENSORS.items():
            d = subset[subset['sensor'] == sensor_key].sort_values('day_of_year')
            if not d.empty:
                ax.plot(d['day_of_year'], d['roi_mean'], marker='o', markersize=2.7,
                        linewidth=1.0, alpha=0.8, color=colors[sensor_key],
                        label=meta['label'])
        ax.set_ylabel(f'{vi} (unitless)')
        ax.set_title(f'June–August {year}')
        ax.grid(alpha=0.25)
        ax.axhline(0, linewidth=0.7, color='0.5')
    axs[-1].set_xlabel('Day of year (nominal observation/composite date)')
    axs[-1].set_xlim(150, 250)
    handles, labels = axs[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.03),
                   ncol=3, fontsize=9, frameon=False)
    fig.suptitle(f'{vi} within-season comparison | {STUDENT_TAG}', fontsize=13)
    path = FIG_DIR / f'within_season_{vi}_all_years.png'
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', path)

## 11. Visual inspection of annual mean maps — five products

This map shows the **June–August mean images in the focus year**. Each map is displayed at its product's nominal scale; resampling may occur for the interactive display. The legend range is held constant for fair visual interpretation. Change `MAP_INDEX` among `'NDVI'`, `'EVI'`, and `'SAVI'` and rerun this cell.

In [ ]:
MAP_INDEX = 'NDVI'  # Change to EVI or SAVI, then rerun.
if MAP_INDEX not in INDICES:
    raise ValueError('MAP_INDEX must be NDVI, EVI, or SAVI.')

vis = {
    'NDVI': {'min': -0.2, 'max': 0.9,
             'palette': ['8c510a', 'd8b365', 'f6e8c3', 'c7eae5', '5ab4ac', '01665e']},
    'EVI': {'min': -0.1, 'max': 0.8,
            'palette': ['8c510a', 'd8b365', 'f6e8c3', 'c7eae5', '5ab4ac', '01665e']},
    'SAVI': {'min': -0.1, 'max': 0.9,
             'palette': ['8c510a', 'd8b365', 'f6e8c3', 'c7eae5', '5ab4ac', '01665e']},
}

# Avoid the same default OpenStreetMap (403) basemap in results visualisation.
annual_map = geemap.Map(center=(49.65, -98.18), zoom=9,
                       basemap='Esri.WorldImagery')
for sensor_key, meta in SENSORS.items():
    img = seasonal_images.get((FOCUS_YEAR, sensor_key))
    if img is not None:
        annual_map.add_layer(img.select(MAP_INDEX), vis[MAP_INDEX],
                             f'{meta["label"]} — {MAP_INDEX} — {FOCUS_YEAR}')
annual_map.add_layer(roi, {'color': 'red'}, 'Student ROI boundary')
annual_map.center_object(roi)
print('Toggle the layer list to inspect products. The last raster added appears on top.')
display(annual_map)

## 12. Export PNG maps directly into **your** mounted Google Drive

Downloads small preview PNGs of the annual June–August means for the focus year. There are up to 15 maps (5 products × 3 indices). These are **rendered maps** for comparison, not numeric analysis rasters. Numeric GeoTIFFs are exported in the next section.

In [ ]:
map_png_log = []
if MAKE_MAP_PNGS:
    for vi in INDICES:
        for sensor_key in SENSORS:
            img = seasonal_images.get((FOCUS_YEAR, sensor_key))
            if img is None:
                continue
            try:
                rendered = img.select(vi).visualize(**vis[vi])
                url = rendered.getThumbURL({
                    'region': roi.bounds(maxError=1), 'dimensions': 600,
                    'format': 'png'
                })
                response = requests.get(url, timeout=120)
                response.raise_for_status()
                path = MAP_DIR / f'{FOCUS_YEAR}_{sensor_key}_{vi}_JuneAugust.png'
                path.write_bytes(response.content)
                map_png_log.append({'year': FOCUS_YEAR, 'sensor': sensor_key,
                                    'index': vi, 'path': str(path), 'success': True})
                print('Saved:', path.name)
            except Exception as err:
                warnings.warn(f'PNG download failed for {sensor_key} / {vi}: {err}')
                map_png_log.append({'year': FOCUS_YEAR, 'sensor': sensor_key,
                                    'index': vi, 'path': '', 'success': False})
    pd.DataFrame(map_png_log).to_csv(CSV_DIR / 'map_png_export_log.csv', index=False)
else:
    print('PNG maps skipped. Set MAKE_MAP_PNGS = True in Section 2 and rerun.')

## 13. Export numeric annual mean GeoTIFFs to your Google Drive (asynchronous)

By default, **one three-band GeoTIFF for each sensor/product and each selected year** is exported. Each file contains numeric June–August mean `NDVI`, `EVI`, and `SAVI` bands. For three years, this creates **15 export tasks** (5 products × 3 years). Tasks are queued by Earth Engine and can take some time. **Colab does not need to remain running once Earth Engine has accepted an export task.**

**Important Earth Engine detail:** `Export.image.toDrive(folder=...)` accepts a **Drive folder name**, not the full nested `MyDrive` path. These GeoTIFFs will therefore appear in a separate **top-level folder** named `P01_<your_tag>_GeoTIFF` in your signed-in Google Drive. Your CSVs, figures, PNGs, ROI and notebook settings are already written into the nested `Satellite_Practicals/Practical_01/<your_tag>/` folder. If you want everything in one place, move the finished TIFF folder in Google Drive afterwards.

For a larger ROI or limited Earth Engine quota, change `START_GEE_GEOTIFF_EXPORTS = False` and export only selected products later. Export uses each product's own nominal scale, without artificially increasing MODIS detail. To reduce task count further, edit the loop below to select particular years or sensors.

In [ ]:
export_tasks = []
if START_GEE_GEOTIFF_EXPORTS:
    for year in YEARS:
        for sensor_key, meta in SENSORS.items():
            img = seasonal_images.get((year, sensor_key))
            if img is None:
                continue
            prefix = f'P01_{STUDENT_TAG}_{year}_{sensor_key}_NDVI_EVI_SAVI_JunAug'
            task = ee.batch.Export.image.toDrive(
                image=img.select(list(INDICES)).toFloat(),
                description=prefix[:100],
                folder=GEE_EXPORT_DRIVE_FOLDER,
                fileNamePrefix=prefix,
                region=roi,
                scale=meta['scale'],
                maxPixels=1e10,
                fileFormat='GeoTIFF'
            )
            task.start()
            export_tasks.append({'year': year, 'sensor': sensor_key,
                                 'indices': ','.join(INDICES), 'task_id': task.id,
                                 'description': prefix, 'task': task})
            print('Queued:', prefix)
else:
    print('GeoTIFF exports disabled. Change START_GEE_GEOTIFF_EXPORTS in Section 2.')

print(f'Queued {len(export_tasks)} export tasks to your own Google Drive folder:',
      GEE_EXPORT_DRIVE_FOLDER)

In [ ]:
# Run again later to check task status. It does NOT download incomplete GeoTIFFs.
if export_tasks:
    statuses = []
    for record in export_tasks:
        status = record['task'].status()
        statuses.append({k: v for k, v in record.items() if k != 'task'} |
                        {'state': status.get('state'),
                         'error_message': status.get('error_message', '')})
    status_df = pd.DataFrame(statuses)
    status_df.to_csv(CSV_DIR / 'geotiff_export_task_status.csv', index=False)
    display(status_df[['year', 'sensor', 'indices', 'state', 'error_message']])
    print('Earth Engine task states are snapshots; rerun this cell to refresh.')
else:
    print('No export tasks have been started in this session.')

## 14. Save reproducibility information and create a convenient ZIP of tabular/figure results

In [ ]:
config = {
    'student_tag': STUDENT_TAG, 'earth_engine_project': EE_PROJECT_ID,
    'years': YEARS, 'focus_year': FOCUS_YEAR, 'season_start': '06-01',
    'season_end_inclusive': '08-31', 'roi_method': ROI_METHOD,
    'roi_area_km2': area_km2, 'roi_geojson': 'student_roi.geojson',
    'sensor_catalogs': {k: v['id'] for k, v in SENSORS.items()},
    'indices': list(INDICES), 's2_max_cloud_percent': S2_MAX_CLOUD_PERCENT,
    'modis_max_summary_qa': MODIS_MAX_SUMMARY_QA,
    's2_timeseries_sampling_scale_m': S2_TIMESERIES_SCALE_M,
    'analysis_notes': [
        'Annual = pixel-wise mean of June-August available VI images followed by ROI mean.',
        'MODIS NDVI/EVI use native MOD13 products; MODIS SAVI uses corresponding reflectance.',
        'MODIS 16-day nominal dates/compositing differ from optical scene acquisitions.',
        'Cloud/QA masking and observation counts differ among sensors and years.',
        'Exported map PNGs are visualizations; GeoTIFFs are numeric analysis rasters.',
    ],
    'generated_utc': datetime.now(timezone.utc).isoformat(),
}
with open(OUTPUT_DIR / 'run_configuration.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

import shutil
zip_stem = MYDRIVE / DRIVE_BASE_FOLDER / 'Practical_01' / f'{STUDENT_TAG}_P01_results'
zip_path = shutil.make_archive(str(zip_stem), 'zip', root_dir=OUTPUT_DIR)
print('Your structured Drive folder:', OUTPUT_DIR)
print('Your ZIP file:', zip_path)
print('GeoTIFFs are exported independently into:', GEE_EXPORT_DRIVE_FOLDER)

## 15. Independent scientific interpretation and deliverables

### Deliverables (all stored in your own Google Drive)
1. Your own saved study area (`student_roi.geojson`) and a short rationale for selecting it, including approximate area and expected land-cover variability.
2. `image_counts_by_sensor_year.csv` and `june_august_annual_roi_means.csv`.
3. `all_observations_june_august_roi_timeseries.csv`.
4. `interannual_3_indices_5_products.png` and all three `within_season_*_all_years.png` figures.
5. The 15 annual mean map PNGs for the focus year (where retrieval succeeds); the three-band numeric GeoTIFFs for all selected years can be submitted when export tasks finish.
6. A **400–600-word interpretation** answering the questions below. Do not use AI to substitute for your independent interpretation of the observed values.

### Questions
- Which sensors/products give the smoothest time series? How much of that difference could result from nominal spatial scale, compositing frequency, and cloud filtering rather than crop growth alone?
- Compare NDVI, EVI and SAVI within each year. Where do the indices agree or diverge, and why could bare-soil background and atmospheric treatment matter?
- Which selected year has the highest June–August mean for each index? Is the ranking stable across products? Support your claims with results from your own ROI.
- Discuss differences in image counts/observation availability. Why is a direct date-to-date comparison between MODIS 16-day composites and Sentinel-2/Landsat scenes imperfect?
- Explain mixed-pixel effects at 250, 500, and 1000 m for your ROI. Would MODIS 1 km be an appropriate product for a single agricultural field? Why or why not?
- Explain limitations of simple arithmetic seasonal means: cloud gaps, uneven sample dates, MODIS compositing, phenological timing, cross-sensor calibration, and residual QA errors.

### Data documentation
- [Sentinel-2 SR Harmonized](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED)
- [Landsat 8 Collection 2 Level 2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2)
- [Landsat 9 Collection 2 Level 2](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC09_C02_T1_L2)
- [MOD13Q1 250 m](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13Q1)
- [MOD13A1 500 m](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13A1)
- [MOD13A2 1 km](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD13A2)
- [Earth Engine Python API / geemap](https://developers.google.com/earth-engine/guides/python_install)
- Huete, A. et al. (2002). Overview of the radiometric and biophysical performance of the MODIS vegetation indices. *Remote Sensing of Environment*, 83, 195–213.
- Huete, A. (1988). A soil-adjusted vegetation index (SAVI). *Remote Sensing of Environment*, 25, 295–309.

**Before submitting:** Check your selected ROI, chosen years, cloud masks, empty images, sensor sampling scales, and whether queued GeoTIFF tasks are marked `COMPLETED`. A task marked `READY` or `RUNNING` is not yet a file in Drive.